In [ ]:
from langchain_demo.config import load_project_environment, require_environment_variable

load_project_environment(override=True)
api_key = require_environment_variable("DEEPSEEK_API_KEY")
base_url = require_environment_variable("DEEPSEEK_API_BASE")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="deepseek:deepseek-flash",  # 最好明确指定供应商
    api_key=api_key,
    base_url=base_url,
)

# zi字符串
print(model.invoke("你好").content)

## 字典列表（推荐，最灵活）

In [ ]:
# 字典列表（推荐，最灵活）
messages = [{"role": "system", "content": "你是诗人，不会数学"}, {"role": "user", "content": "帮我计算1+1等于多少"}]

print(model.invoke(messages).content)

## 多轮对话

In [ ]:
# 默认情况下，如果不传递历史，AI就会失忆
messages = [
    {"role": "system", "content": "你是一个专业的数学老师"},
    {"role": "user", "content": "1+1=？"},
    {"role": "assistant", "content": "3"},
    {"role": "user", "content": "我刚刚问了什么问题"},
]

print(model.invoke(messages).content)

### 传递记忆

In [ ]:
messages = [{"role": "system", "content": "你是一个很凶的助手"}, {"role": "user", "content": "我是主人"}]

response = model.invoke(messages)
print("ai 回复：", response.content)

# 添加记忆，将上次的对话结果追加
messages.append({"role": "assistant", "content": response.content})
messages.append({"role": "user", "content": "你能为我做什么"})

response = model.invoke(messages)
print("ai 回复：", response.content)

## 消息对象列表输入

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [SystemMessage(content="你是很凶的AI助手"), HumanMessage(content="我是主人")]
response = model.invoke(messages)
print(response.content)

messages.append(AIMessage(content=response.content))
messages.append(HumanMessage(content="我是谁"))
response = model.invoke(messages)
print(response.content)

### invoke的返回值

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [SystemMessage(content="你是很凶的AI助手"), HumanMessage(content="我是主人")]
response = model.invoke(messages)
print(type(response))  # <class 'langchain_core.messages.ai.AIMessage'>

### 美化输出，获取ai响应的结构数据

In [ ]:
# 使用rich的print美化输出
from rich import print as rprint

rprint(response)

response_metadata = response.response_metadata
print(f"模型提供商{response_metadata['model_provider']}")
print(f"使用模型{response_metadata['model_name']}")
print(f"结束原因{response_metadata['finish_reason']}")  # stop正常结束，length表示因长度受限结束

# token使用情况
usage = response_metadata["token_usage"]
print(f"输入token{usage['prompt_tokens']}")
print(f"输出token{usage['completion_tokens']}")
print(f"总计token{usage['total_tokens']}")

# 消息ID
print(f"消息ID：{response.id}")